# County Weather Point Selection

This notebook shows how county-level latitude and longitude points are selected for weather downloads. The resulting county weather is later population-weighted into BA-level weather for TELL.

There are two stages. First, calculate the population-weighted county centroid from U.S. Census data. Second, match that centroid to the nearest WTK, BC-HRRR, and NSRDB grid points. The resulting grid IDs (`gid`) are used by `county_hsds_download_and_ba_weather_aggregation.ipynb` to download county weather.

The motivation for identifying the population-weighted county centroid and downloading weather data at that point is because weather conditions at the population-weighted county centroid are more likely to be representative of the weather experienced by the majority of the population in that county, and therefore, the weather responsible for driving the majority of load in that county. As such, there should be a closer correlation between load and the weather at the population-weighted centroid as opposed to a random lat/lon in the county or the geographic county centroid. 

County centroids were computed from raw 2020 Census P.L. 94-171 block-level population counts and the 2020 TIGER/Line Block20 geodatabase. Those national inputs are very large, so this public notebook uses a small Arthur County, Nebraska (FIPS code 31005) example with simplified CSV inputs. `"arthur_county_block_population_2020.csv"` corresponds to the 2020 Census P.L. 94-171 block-level population counts, while `"arthur_county_tiger_block_points_2020.csv"` corresponds to the geodatabase and gives the lat/lon of each census block. By joining the two datasets on a common column, we can get the population and lat/lon of each census block, and from those census blocks, calculate the population-weighted lat/lon centroid of the county.

## Imports And Package Setup

In [1]:
from pathlib import Path

import pandas as pd
from rex import Resource  # Step 5: read HSDS resource metadata such as grid lat/lon.
from scipy.spatial import cKDTree  # Step 5: nearest-neighbor search from county centroid to grid point.

DATA_ROOT = Path("../../data")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        "Expected to find the packaged data folder at ../data. "
        "Open this notebook from inside notebooks/data_flow."
    )


## Steps 1–3: Build The Block-Level Centroid Inputs

### Step 1: Read Census Block Population

The population side of the calculation comes from 2020 Census block records. It keeps each block identifier as a 15-digit text `GEOID20` so leading zeros are preserved before merging with TIGER coordinates. It uses the Census block total-population field `POP100` directly.

### Step 2: Read TIGER Block Internal Points

The coordinate side comes from TIGER Block20 internal points. The packaged example keeps the same `GEOID20` plus the block internal-point latitude and longitude.

### Step 3: Merge Blocks And Extract County FIPS

The block population and TIGER internal-point tables join by `GEOID20`. For a 2020 Census block GEOID, the first 5 characters are the full county FIPS code: first 2 digits for state, followed by 3 digits for county within that state.


In [2]:
ARTHUR_EXAMPLE_DIR = DATA_ROOT / "county_weather" / "arthur_county_point_selection_example"
ARTHUR_BLOCK_POPULATION_CSV = ARTHUR_EXAMPLE_DIR / "arthur_county_block_population_2020.csv"
ARTHUR_TIGER_POINTS_CSV = ARTHUR_EXAMPLE_DIR / "arthur_county_tiger_block_points_2020.csv"

# Step 1
block_population = pd.read_csv(ARTHUR_BLOCK_POPULATION_CSV, dtype={"GEOID20": str})
print(f"Arthur County block rows: {len(block_population)}")
display(block_population.head())

# Step 2
tiger_points = pd.read_csv(ARTHUR_TIGER_POINTS_CSV, dtype={"GEOID20": str})
print(f"Arthur County TIGER rows: {len(tiger_points)}")
display(tiger_points.head())

# Step 3
blocks = block_population.merge(tiger_points, on="GEOID20", validate="one_to_one")

# GEOID20 starts with state FIPS + county code; the first 5 characters are county FIPS.
blocks["county_fips"] = blocks["GEOID20"].str[:5]
blocks

Arthur County block rows: 46


,GEOID20,POP100
0,310059583001000,6
1,310059583001001,4
2,310059583001002,8
3,310059583001003,2
4,310059583001004,2


Arthur County TIGER rows: 46


,GEOID20,INTPTLAT,INTPTLON
0,310059583001000,41.730981,-101.438985
1,310059583001001,41.703768,-101.476879
2,310059583001002,41.734505,-101.577511
3,310059583001003,41.704448,-101.636337
4,310059583001004,41.731602,-101.694203


,GEOID20,POP100,INTPTLAT,INTPTLON,county_fips
0,310059583001000,6,41.730981,-101.438985,31005
1,310059583001001,4,41.703768,-101.476879,31005
2,310059583001002,8,41.734505,-101.577511,31005
3,310059583001003,2,41.704448,-101.636337,31005
4,310059583001004,2,41.731602,-101.694203,31005
5,310059583001005,24,41.702410,-101.844856,31005
6,310059583001006,7,41.642200,-101.975206,31005
7,310059583001007,2,41.624178,-101.877503,31005
8,310059583001008,0,41.656020,-101.796020,31005
9,310059583001009,18,41.618735,-101.728888,31005


## Step 4: Population-Weighted County Centroid

For county `c`, the population-weighted centroid is:

`county_pop_lat_c = sum(POP100_b * INTPTLAT_b) / sum(POP100_b)`

`county_pop_lon_c = sum(POP100_b * INTPTLON_b) / sum(POP100_b)`

where `b` is a Census block in county `c`, `POP100_b` is block population from P.L. 94-171, and `INTPTLAT_b`/`INTPTLON_b` are TIGER block internal-point coordinates.


In [3]:
# Weight each block coordinate by that block's population.
blocks["POP100_times_INTPTLAT"] = blocks["POP100"] * blocks["INTPTLAT"]
blocks["POP100_times_INTPTLON"] = blocks["POP100"] * blocks["INTPTLON"]

# Sum population and weighted coordinates within each county FIPS.
county_centroid_example = blocks.groupby("county_fips", as_index=False).agg(population=("POP100", "sum"), sum_population_times_lat=("POP100_times_INTPTLAT", "sum"), sum_population_times_lon=("POP100_times_INTPTLON", "sum"))

# Divide weighted coordinate sums by total county population.
county_centroid_example["county_pop_lat"] = (county_centroid_example["sum_population_times_lat"] / county_centroid_example["population"])
county_centroid_example["county_pop_lon"] = (county_centroid_example["sum_population_times_lon"] / county_centroid_example["population"])
county_centroid_example

,county_fips,population,sum_population_times_lat,sum_population_times_lon,county_pop_lat,county_pop_lon
0,31005,434,18041.356747,-44128.760654,41.569946,-101.679172


## Step 5: Match The County Centroid To HSDS Grid Points

Nearest-grid matching uses HSDS grid metadata only: `gid`, grid latitude, and grid longitude. It does **not** download weather time series. The actual weather-variable reads happen in `county_hsds_download_and_ba_weather_aggregation.ipynb`.

Before running this step, start the local HSDS service at `http://localhost:5101` with access to the `nrel-pds-hsds` bucket.

This step reads metadata from one representative historical resource file for each weather source with `rex.Resource`, builds a `cKDTree` from the grid coordinates, and selects the nearest grid point for the example county centroid. Each weather source is matched separately because WTK, BC-HRRR, and NSRDB use different grids.


In [4]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"

# Use one representative resource year for each source's historical grid.
RESOURCE_PATHS = {
    "wtk": "/nrel/wtk/conus/wtk_conus_2013.h5",
    "bchrrr": "/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5",
    "nsrdb": "/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5",
}

centroid = county_centroid_example.iloc[0]
county_point = centroid[["county_pop_lat", "county_pop_lon"]].to_numpy(dtype=float)
selected_rows = []

for weather_source, resource_path in RESOURCE_PATHS.items():
    # 1. Read the latitude/longitude grid for this weather source.
    with Resource(resource_path, hsds=True, hsds_kwargs={"endpoint": HSDS_ENDPOINT, "api_key": HSDS_API_KEY, "bucket": HSDS_BUCKET}) as resource:
        grid_coordinates = resource.coordinates

    # 2. The coordinate row number is the HSDS grid id.
    tree = cKDTree(grid_coordinates)
    _, selected_gid = tree.query(county_point)
    selected_gid = int(selected_gid)
    grid_lat, grid_lon = grid_coordinates[selected_gid]

    # 3. Store the selected grid id and coordinates for this weather source.
    selected_rows.append({
        "county_fips": centroid["county_fips"],
        "weather_source": weather_source,
        "resource_path": resource_path,
        "selected_gid": selected_gid,
        "county_pop_lat": float(centroid["county_pop_lat"]),
        "county_pop_lon": float(centroid["county_pop_lon"]),
        "grid_lat": float(grid_lat),
        "grid_lon": float(grid_lon),
    })

nearest_grid_points = pd.DataFrame(selected_rows)
nearest_grid_points


,county_fips,weather_source,resource_path,selected_gid,county_pop_lat,county_pop_lon,grid_lat,grid_lon
0,31005,wtk,/nrel/wtk/conus/wtk_conus_2013.h5,1014962,41.569946,-101.679172,41.575672,-101.680725
1,31005,bchrrr,/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5,1014962,41.569946,-101.679172,41.575672,-101.680725
2,31005,nsrdb,/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5,571348,41.569946,-101.679172,41.570000,-101.660004


## Downstream Use

The packaged file `county_centroid_regrid.csv` is the county-level weather point list used by `county_hsds_download_and_ba_weather_aggregation.ipynb`.

That downstream notebook reads weather variables at each selected `gid`, then population-weights county weather into BA and state-level weather.
